In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [2]:
X, y = load_diabetes(return_X_y = True)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 2)

In [4]:
print(f"Shape of feature {X.shape}, shape of the response {y.shape}")

Shape of feature (442, 10), shape of the response (442,)


In [9]:
sk_ols = LinearRegression()
sk_ols.fit(X = X_train, y = y_train)
print(f"{sk_ols.intercept_}, {sk_ols.coef_}")
print("R2 Score", r2_score(y_test, sk_ols.predict(X_test)))

151.88331005254167, [  -9.15865318 -205.45432163  516.69374454  340.61999905 -895.5520019
  561.22067904  153.89310954  126.73139688  861.12700152   52.42112238]
R2 Score 0.4399338661568968


In [6]:
class My_SGDRegressor:
    def __init__(self, learning_rate: float, epochs: int) -> None:
        self.lr = learning_rate
        self.epochs = epochs
        self.intercept_ = 0
        self.coef_ = np.array([])
    
    def fit(self, X_train, y_train):
        X_train = np.insert(X_train, 0, 1, axis=1)
        rows, columns = X_train.shape
    
        theta = np.zeros(columns)  # 1D
    
        for _ in range(self.epochs):
            for _ in range(rows):
                idx = np.random.randint(0, rows)
                x_i = X_train[idx]
                y_i = y_train[idx]
    
                y_pred = np.dot(x_i, theta)
                error = y_pred - y_i
    
                gradient = 2 * error * x_i
                theta -= self.lr * gradient
    
        self.intercept_ = theta[0]
        self.coef_ = theta[1:]

        self.intercept_ = theta[0]
        self.coef_ = theta[1 : ]

    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_

    def alternate_fit(self,X_train,y_train):
        # init your coefs
        self.alternate_intercept_ = 0
        self.alternate_coef_ = np.ones(X_train.shape[1])
        
        for i in range(self.epochs):
            for j in range(X_train.shape[0]):
                idx = np.random.randint(0,X_train.shape[0])
                
                y_hat = np.dot(X_train[idx],self.alternate_coef_) + self.alternate_intercept_
                
                alternate_intercept_der = -2 * (y_train[idx] - y_hat)
                self.alternate_intercept_ = self.alternate_intercept_ - (self.lr * alternate_intercept_der)
                
                alternate_coef_der = -2 * np.dot((y_train[idx] - y_hat),X_train[idx])
                self.alternate_coef_ = self.alternate_coef_ - (self.lr * alternate_coef_der)
        print(self.intercept_,self.coef_)

    def alternate_predict(self, X_test):
         return np.dot(X_test, self.alternate_coef_) + self.alternate_intercept_

In [7]:
my_sgdr = My_SGDRegressor(learning_rate = 0.001, epochs = 200)

my_sgdr.fit(X_train = X_train, y_train = y_train)

print(f"My SGD: {my_sgdr.intercept_}, {my_sgdr.coef_}")

print("R2 Score", r2_score(y_test, my_sgdr.predict(X_test)))

my_sgdr.alternate_fit(X_train = X_train, y_train = y_train)

print("R2 Score", r2_score(y_test, my_sgdr.alternate_predict(X_test)))


My SGD: 150.30094232663163, [  57.18860261   -8.38768222  209.69036588  158.02674773   43.06064328
   21.46171282 -119.34470474  110.14326152  198.23179065  105.82768501]
R2 Score 0.3614641316134223
150.30094232663163 [  57.18860261   -8.38768222  209.69036588  158.02674773   43.06064328
   21.46171282 -119.34470474  110.14326152  198.23179065  105.82768501]
R2 Score 0.3628247268459638


In [8]:
sk_sgdr = SGDRegressor(max_iter = 500, learning_rate = 'constant')
sk_sgdr.fit(X = (X_train), y= y_train)
print(f"Scikit SGD: {sk_sgdr.intercept_}, {sk_sgdr.coef_}")
print("R2 Score", r2_score(y_test, sk_sgdr.predict(X_test)))

Scikit SGD: [151.62357269], [  54.22999877  -72.03445179  358.14494307  251.95882331   14.81583105
  -32.96682454 -174.90713459  127.83372649  325.10520357  128.08306177]
R2 Score 0.43595052499431153


Learning Schedule: